<a href="https://colab.research.google.com/github/eshwar-2006/flyrank-ml-internship/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eshwar-2006/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


**My chosen lane: ranking/scoring.** The practical decision is: *which content pages should an editor review first for a possible content refresh?* I will create a priority score from available page-level signals and rank pages so the team can inspect the highest-priority items first. This is a decision-support tool, not an automatic instruction to edit a page. The starter label is only a proxy for current decline, so I will describe results as directional rather than as proof of future performance.


In [1]:
# Frame the task and verify that the starter dataset has the columns we expect.
task_type = "ranking / scoring"
decision = "Prioritize content pages for human review"
required_columns = {"content_id", "client_id", "trend_direction", "trend_pct",
                    "ctr", "impressions_90d", "sessions_90d"}

print("Task type:", task_type)
print("Decision supported:", decision)
print("Expected key columns:", sorted(required_columns))


Task type: ranking / scoring
Decision supported: Prioritize content pages for human review
Expected key columns: ['client_id', 'content_id', 'ctr', 'impressions_90d', 'sessions_90d', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


The ideal target would be a future observed outcome showing whether a page's performance declines or recovers in a later time window. The starter slice does not provide that future outcome. For this framing exercise, I will use `is_declining_label` as a **proxy** when it is available, or derive the same proxy as `trend_direction == "down"`. This label is rule-derived from the current trend, not an independent future outcome. Therefore, I must not use `trend_direction` or `trend_pct` as model features, and I must not claim that the proxy proves future decline.


In [2]:
# Load the anonymized starter CSV from the repository when available.
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path.cwd() / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data/raw/content_refresh_anonymized.csv",
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    print("Loaded local starter data from:", data_path)
else:
    # Public, anonymized starter CSV; read into memory without saving/committing the dataset.
    data_url = (
        "https://raw.githubusercontent.com/flyrank-bih/"
        "flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(data_url)
    print("Loaded the public anonymized starter CSV into memory.")

print("Rows:", len(df), "| Columns:", len(df.columns))
print("Required columns present:", required_columns.issubset(df.columns))

if "trend_direction" not in df.columns:
    raise ValueError("Expected column 'trend_direction' is missing; check the dataset version.")

df["is_declining_label"] = (df["trend_direction"].astype(str) == "down").astype(int)
display(df[["trend_direction", "is_declining_label"]].head())
print("\nProxy-label counts:")
display(df["is_declining_label"].value_counts(dropna=False).rename_axis("is_declining_label").to_frame("rows"))


Loaded the public anonymized starter CSV into memory.
Rows: 30000 | Columns: 44
Required columns present: True


,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1



Proxy-label counts:


,rows
is_declining_label,
1,16262
0,13738


## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Primary metric: Precision@K, with K = 20 and K = 50.** After sorting pages by a priority score, Precision@K is the fraction of the top K pages that have the proxy label `is_declining_label = 1`. This matches the human-review decision: how many of the first K pages flagged for review are labelled as declining? I will compare a transparent rule-based ranking with the overall proxy-label base rate. Beating the base rate is a useful check, but it does not establish that the label predicts future outcomes.


In [3]:
# Define the metric before comparing ranking rules.
def precision_at_k(frame, score_col, label_col="is_declining_label", k=20):
    ranked = frame.sort_values(score_col, ascending=False, na_position="last").head(k)
    if len(ranked) == 0:
        return float("nan")
    return float(ranked[label_col].mean())

label_rate = float(df["is_declining_label"].mean())
print(f"Proxy-label base rate: {label_rate:.3f} ({label_rate:.1%})")

# A simple, transparent review rule: prioritize pages with more search impressions.
# This is only a baseline for comparison, not a trained model.
if "impressions_90d" not in df.columns:
    raise ValueError("Expected 'impressions_90d' column is missing.")

for k in (20, 50):
    actual_k = min(k, len(df))
    p_at_k = precision_at_k(df, "impressions_90d", k=actual_k)
    print(f"Rule baseline Precision@{actual_k}: {p_at_k:.3f}")


Proxy-label base rate: 0.542 (54.2%)
Rule baseline Precision@20: 0.450
Rule baseline Precision@50: 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


One row represents one pseudonymized content item/page, with aggregated activity metrics for a trailing 90-day window. `content_id` and `client_id` are identifiers for grouping or splitting only, never model features. I will inspect the actual columns, data types, missingness, and sample rows. The data dictionary notes that `trend_direction` and `trend_pct` define the proxy label and must be excluded from model features; identifiers must also be excluded.


In [4]:
# Inspect the real dataframe and show a small, safe preview.
print("Dataframe shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

preview_cols = [c for c in [
    "content_id", "content_type", "impressions_90d", "sessions_90d",
    "ctr", "avg_position", "trend_direction", "is_declining_label"
] if c in df.columns]
display(df[preview_cols].head(10))

print("\nData types and missing values for selected columns:")
display(pd.DataFrame({
    "dtype": df[preview_cols].dtypes.astype(str),
    "missing_count": df[preview_cols].isna().sum(),
    "missing_pct": (df[preview_cols].isna().mean() * 100).round(2)
}))


Dataframe shape: (30000, 45)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


,content_id,content_type,impressions_90d,sessions_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,keyword article,3803,17,0.76,10.6,down,1
1,content_a1fb4e703a9e,keyword article,15320,9,0.05,20.3,down,1
2,content_9aa793d4d895,keyword article,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,keyword article,11751,78,0.49,6.2,stable,0
4,content_d99b7a2d90ca,keyword article,19140,145,0.13,44.0,down,1
5,content_d4084a4bc775,keyword article,3970,5,0.03,8.5,down,1
6,content_9a34b442b552,keyword article,20,1,0.00,7.0,down,1
7,content_a63219c6e95a,keyword article,1724,28,0.06,21.2,stable,0
8,content_5e6c160719bc,keyword article,32574,68,0.09,46.0,down,1
9,content_c27558df2b0c,keyword article,1240,3,0.16,4.9,down,1



Data types and missing values for selected columns:


,dtype,missing_count,missing_pct
content_id,object,0,0.0
content_type,object,0,0.0
impressions_90d,int64,0,0.0
sessions_90d,int64,0,0.0
ctr,float64,0,0.0
avg_position,float64,0,0.0
trend_direction,object,0,0.0
is_declining_label,int64,0,0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A single rule such as sorting only by impressions cannot account for different combinations of click-through rate, traffic, content type, freshness, and engagement. A model may combine multiple signals to produce a more useful review priority. However, ML only earns its place if it improves ranking quality on data not used to build it. A later model evaluation should use a client-level holdout where possible, compare Precision@K with a simple baseline on the same split, and exclude `trend_direction`, `trend_pct`, `content_id`, and `client_id` from features. Because the current label is rule-derived from the same time window, the result is a proxy-label exercise, not evidence of future decline prediction.


In [5]:
# Demonstrate a fixed-rule baseline and a second transparent rule.
# These are baseline comparisons only; they do not train an ML model.
baseline = df.copy()

# Lower CTR may identify pages worth inspecting, but CTR can be noisy at low volume.
if "ctr" in baseline.columns:
    baseline["low_ctr_review_score"] = -pd.to_numeric(baseline["ctr"], errors="coerce")
else:
    raise ValueError("Expected 'ctr' column is missing.")

# Keep the rules interpretable and report the same metric for each.
for score_col, description in [
    ("impressions_90d", "Higher impressions first"),
    ("low_ctr_review_score", "Lower CTR first"),
]:
    print("\nRule:", description)
    for k in (20, 50):
        actual_k = min(k, len(baseline))
        value = precision_at_k(baseline, score_col, k=actual_k)
        print(f"  Precision@{actual_k}: {value:.3f}")

print("\nInterpretation: compare these values with the proxy-label base rate,")
print("but do not claim future predictive performance from this cross-sectional proxy.")



Rule: Higher impressions first
  Precision@20: 0.450
  Precision@50: 0.420

Rule: Lower CTR first
  Precision@20: 0.450
  Precision@50: 0.580

Interpretation: compare these values with the proxy-label base rate,
but do not claim future predictive performance from this cross-sectional proxy.


## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
